In [1]:
!pip -q install -U langchain langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GROQ_API_KEY"]=input("enter your api key=")
print("groq api key configured")

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM created!")

LLM created!


In [10]:
from langchain_core.tools import tool
@tool
def calculate_trip_cost(days:int,daily_budget:int)->str:
  """calculate the estimated total cost of a trip."""
  total=days*daily_budget
  return f"your total trip cost is {total}"
  print(calculate_trip_cost.invoke({
      "days":5,
      "daily_budget":3000
  }))


In [11]:
@tool
def get_destination_info(destination: str) -> str:
    """Provide basic information about a travel destination."""

    destinations = {
        "goa": "Beaches, nightlife, water sports and Portuguese heritage.",
        "manali": "Mountains, snow activities, trekking and scenic landscapes.",
        "jaipur": "Forts, palaces, Rajasthani culture and historical sites."
    }

    return destinations.get(
        destination.lower(),
        "No information available for this destination."
    )

print(get_destination_info.invoke({
    "destination": "Goa"
}))

Beaches, nightlife, water sports and Portuguese heritage.


In [12]:
from langchain.agents import create_agent
travel_agent=create_agent(
    model=llm,
    tools=[
        calculate_trip_cost,
        get_destination_info
    ]
)
print("travel agent created")

travel agent created


In [13]:
result = travel_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is Goa known for?"
        }
    ]
})

print(result["messages"][-1].content)

Goa, a tiny state on India’s western coast, is famous for a blend of natural beauty, cultural heritage, and a laid‑back vibe that draws travelers from around the world. Here are the highlights that put Goa on the map:

| Category | What Goa is Known For |
|----------|-----------------------|
| **Beaches** | • **Golden sands & turquoise waters** – Calangute, Baga, Anjuna, Vagator, Palolem, Agonda, Morjim, and many more.<br>• **Beach parties & sunset shacks** – especially in North Goa’s party hubs. |
| **Nightlife** | • **Clubs & bars** – Tito’s, Mambo’s, Café Mambo, Club Cubana, and the legendary *Sunburn* music festival.<br>• **Live music & DJ events** – from reggae to EDM to Goan folk. |
| **Portuguese Heritage** | • **Architecture** – Baroque churches (Basilica of Bom Bom, Se Cathedral, Church of St Francis of Assisi), colorful Portuguese houses, and the UNESCO‑listed **Old Goa**.<br>• **Cultural fusion** – a mix of Indian and Portuguese customs, language (Konkani with Portuguese loa

In [14]:
for message in result["messages"]:
    print("TYPE:", type(message).__name__)
    print("CONTENT:", message.content)
    print("TOOL CALLS:", getattr(message, "tool_calls", None))
    print("-" * 60)

TYPE: HumanMessage
CONTENT: What is Goa known for?
TOOL CALLS: None
------------------------------------------------------------
TYPE: AIMessage
CONTENT: Goa, a tiny state on India’s western coast, is famous for a blend of natural beauty, cultural heritage, and a laid‑back vibe that draws travelers from around the world. Here are the highlights that put Goa on the map:

| Category | What Goa is Known For |
|----------|-----------------------|
| **Beaches** | • **Golden sands & turquoise waters** – Calangute, Baga, Anjuna, Vagator, Palolem, Agonda, Morjim, and many more.<br>• **Beach parties & sunset shacks** – especially in North Goa’s party hubs. |
| **Nightlife** | • **Clubs & bars** – Tito’s, Mambo’s, Café Mambo, Club Cubana, and the legendary *Sunburn* music festival.<br>• **Live music & DJ events** – from reggae to EDM to Goan folk. |
| **Portuguese Heritage** | • **Architecture** – Baroque churches (Basilica of Bom Bom, Se Cathedral, Church of St Francis of Assisi), colorful Port

In [16]:
result=travel_agent.invoke({
    "messages":[
        {
        "role":"user",
        "content":"""
        i want to visit goa for 5 days.
        tell me what is goa known for and calculate
        my approximate cost if i spend 3000 rupees per day.
        """
        }
    ]
})
print(result["messages"][-1].content)

**Goa – What It’s Known For**

- **Stunning Beaches** – From the lively Baga and Calangute to the serene Palolem and Agonda, Goa’s coastline offers something for every beach lover.  
- **Vibrant Nightlife** – Beach shacks, clubs, and bars keep the party going well into the night, especially in places like Tito’s, Café Mambo, and Club Cabana.  
- **Water Sports & Adventure** – Jet‑skiing, parasailing, windsurfing, scuba diving, and dolphin‑watching are popular activities along the coast.  
- **Portuguese Heritage** – Charming colonial architecture, historic forts (Aguada, Chapora), and beautiful churches such as Basilica of Bom Jesus and Se Cathedral reflect Goa’s Portuguese past.  
- **Delicious Cuisine** – Fresh seafood, Goan curries, vindaloo, xacuti, and the famous feni (cashew or coconut spirit).  
- **Festivals & Culture** – Carnival, Shigmo, and numerous music festivals add a cultural flair to the visit.

**Estimated Cost for Your 5‑Day Trip**

- **Daily budget:** ₹3,000  
- **Nu

In [17]:
for message in result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for tool_call in message.tool_calls:
            print("Tool:", tool_call["name"])
            print("Arguments:", tool_call["args"])
            print("-" * 40)

Tool: get_destination_info
Arguments: {'destination': 'Goa'}
----------------------------------------
Tool: calculate_trip_cost
Arguments: {'daily_budget': 3000, 'days': 5}
----------------------------------------


In [18]:
travel_agent = create_agent(
    model=llm,
    tools=[
        calculate_trip_cost,
        get_destination_info
    ],
    system_prompt="""
    You are a helpful travel planning assistant.

    Your responsibilities:
    - Help users plan simple trips.
    - Use destination information tools when appropriate.
    - Use the cost calculator when calculating trip expenses.
    - You may use multiple tools for a single request.
    - Give the user a clear and concise final answer.
    """
)

print("Travel agent updated!")


Travel agent updated!


In [19]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

travel_agent = create_agent(
    model=llm,
    tools=[
        calculate_trip_cost,
        get_destination_info
    ],
    system_prompt="You are a helpful travel planning assistant.",
    checkpointer=memory
)

config = {
    "configurable": {
        "thread_id": "travel_1"
    }
}

print("Memory enabled!")

Memory enabled!


In [20]:
result = travel_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I am planning a 5-day trip to Goa."
            }
        ]
    },
    config
)

result = travel_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How many days am I staying?"
            }
        ]
    },
    config
)

print(result["messages"][-1].content)


You mentioned that you’re planning a **5‑day** trip to Goa. Let me know if you’d like any help estimating the budget, planning a day‑by‑day itinerary, or anything else for those five days!


In [21]:
try:
    result = travel_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Calculate the cost of a 7-day trip with a daily budget of ₹2500."
                }
            ]
        },
        config
    )

    print(result["messages"][-1].content)

except Exception as e:
    print("Travel agent error:", e)

Based on a daily budget of **₹2,500**, a **7‑day** trip would cost approximately **₹17,500** in total. Let me know if you’d like help breaking that down (e.g., accommodation, food, transport) or planning your itinerary!


In [22]:
result = travel_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                I'm planning a 5-day trip to Goa.

                Tell me:
                1. What Goa is known for
                2. My estimated cost if I spend ₹3000 per day
                3. Give me a short recommendation for the trip
                """
            }
        ]
    },
    config
)

print(result["messages"][-1].content)

**1. What Goa is known for**  
- **Sun‑kissed beaches** – Baga, Calangute, Anjuna, Palolem, Agonda, and many quieter coves.  
- **Vibrant nightlife** – beach shacks, clubs, and sunset parties (e.g., Café Mambo, Tito’s, Club Cabana).  
- **Portuguese heritage** – colorful churches (Basilica of Bom Jesus, Se Cathedral), forts (Aguada, Chapora), and colonial‑style architecture.  
- **Spice & seafood cuisine** – Goan fish curry, prawn balchão, vindaloo, and the famous feni (cashew or coconut spirit).  
- **Water‑sports & adventure** – jet‑skiing, parasailing, scuba diving, and dolphin‑watching tours.  
- **Relaxed vibe & yoga retreats** – many beachfront yoga centers and wellness resorts.

**2. Your estimated cost**  
- Daily budget: **₹3,000**  
- Duration: **5 days**  
- **Total estimated cost:** **₹15,000** (covers accommodation, meals, local transport, and modest activities).

**3. Short recommendation for the trip**  

- **Day 1:** Arrive in Panaji, settle into a budget‑friendly guest

In [23]:
for message in result["messages"]:
    print("TYPE:", type(message).__name__)

    if message.content:
        print("CONTENT:", message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("TOOL CALLS:", message.tool_calls)

    print("-" * 60)

TYPE: HumanMessage
CONTENT: I am planning a 5-day trip to Goa.
------------------------------------------------------------
TYPE: AIMessage
CONTENT: That sounds like a wonderful getaway! 🌴

Here are a few quick things that will help me give you the most useful recommendations:

1. **What’s your daily budget?** (e.g., budget, mid‑range, or luxury) – I can calculate an estimated total cost for the 5 days.  
2. **What are your interests?** (beaches, nightlife, heritage sites, nature, water sports, food, etc.) – I can suggest must‑see spots and activities that match your vibe.  
3. **Do you need help with anything else?** (accommodation ideas, transport options, itinerary layout, packing tips, etc.)

Let me know, and I’ll pull together the details for your Goa adventure!
------------------------------------------------------------
TYPE: HumanMessage
CONTENT: How many days am I staying?
------------------------------------------------------------
TYPE: AIMessage
CONTENT: You mentioned that 

In [24]:
result = travel_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                Actually, forget Goa for this trip.
                I want to visit Manali for 4 days instead.
                Tell me what Manali is known for and calculate
                the cost with a daily budget of ₹2500.
                """
            }
        ]
    },
    config
)

print(result["messages"][-1].content)

**What Manali is known for**

- **Stunning mountain scenery** – nestled in the Himalayas, with snow‑capped peaks, pine forests, and the Beas River winding through town.  
- **Adventure sports** – paragliding, river rafting, trekking (e.g., Solang Valley, Rohtang Pass), and skiing in winter.  
- **Cool climate** – a refreshing escape from the heat, especially pleasant from September to May.  
- **Cultural sites** – Hadimba Temple (wooden architecture), Manu Temple, Vashisht Hot Springs, and the Tibetan Monastery at nearby **Naggar**.  
- **Apple orchards & local produce** – fresh apples, honey, and Himachali cuisine (siddu, trout, thukpa).  
- **Shopping & nightlife** – Mall Road for souvenirs, woolen clothing, and cozy cafés.

**Estimated cost for a 4‑day trip**

- Daily budget: **₹2,500**  
- Duration: **4 days**  
- **Total estimated cost:** **₹10,000** (covers accommodation, meals, local transport, and modest activities).

Let me know if you’d like a suggested 4‑day itinerary, accom

In [25]:
result = travel_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What makes a good travel itinerary?"
            }
        ]
    },
    config
)

print(result["messages"][-1].content)

### What Makes a Good Travel Itinerary?

A well‑crafted itinerary is the backbone of a stress‑free, enjoyable trip. It doesn’t have to be a rigid schedule, but it should give you enough structure to make the most of your time while leaving room for spontaneity. Below are the key ingredients that turn a bland list of “things to see” into a smooth, memorable travel plan.

| # | Element | Why It Matters | How to Do It |
|---|----------|----------------|--------------|
| **1** | **Clear Goal & Priorities** | Helps you focus on what truly matters to you (culture, nature, food, adventure, relaxation). | Write a short “trip purpose” statement and list 3‑5 must‑see attractions or experiences. |
| **2** | **Realistic Time Allocation** | Prevents burnout and missed connections. Over‑packing a day leads to fatigue; under‑packing wastes time. | Estimate travel time between spots (use Google Maps or local transport apps). Add a buffer of 15‑30 min per move. |
| **3** | **Logical Geographic Flow** |